# CellposeSAM segmentation across an AVITI24 cytoprofiling run

Run [CellposeSAM (CPSAM)](https://github.com/mouseland/cellpose) across every tile in a Teton or Teton Atlas run and write the segmentation masks Cells2Stats expects. CPSAM is a general-purpose model that integrates the [Segment Anything Model](https://segment-anything.com/) architecture; use it when your cell type is not represented in the Element Biosciences model library or when the General Element Biosciences model produces poor results even after diameter tuning.

This notebook is the companion to the [Custom segmentation tutorial](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/). Read the tutorial first for the full context: run-type identification, when to use CPSAM, and post-segmentation Cells2Stats re-run.

## Prerequisites

Confirm the following before running this notebook:

- Your run is a **Teton** or **Teton Atlas** run. CPSAM requires the actin channel and fails on Cell Paint only runs because no actin `.tif` file exists. For Cell Paint only runs, use an Element Biosciences 2-channel model and follow [Run a tile evaluation](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#tile-evaluation) and [Run full segmentation](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#full-segmentation).
- You created the separate `cpsam` Python environment with Cellpose 4.x installed from the MouseLand GitHub HEAD. See [Set up the CPSAM environment](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#cpsam-setup-env). Do not install Cellpose 4.x into your `cytoprofiling-seg` environment.
- The `cpsam` environment is selected as this notebook's kernel (top menu: **Kernel → Change kernel → CellposeSAM**).
- A GPU is available. CPSAM on CPU is prohibitively slow for full-run processing.

> **First run downloads ~1.15 GB.** The first time you initialize the CPSAM model, Cellpose automatically downloads the model weights (~1.15 GB) from HuggingFace to `~/.cellpose/models/`. Subsequent runs use the cached weights and do not require an internet connection.

## Step 1 — Import packages

Load the imaging, numerics, and Cellpose packages used throughout the rest of the notebook.

In [1]:
import json
import os

import numpy as np
import skimage
from cellpose import core, models, transforms


## Step 2 — Provide Input and Output Paths

Set the two required paths. Use a fresh `output_location` per re-segmentation pass so CPSAM masks do not overwrite Element Biosciences masks.

In [2]:
# Edit both paths before running the rest of the notebook.

# Path to your AVITI24 run output folder
run_directory   = r"/path/to/your/Run/Output/Folder"

# Where to write the CPSAM segmentation mask outputs (must be a different folder)
output_location = r"/path/to/your/Run/Output/Folder/Segmentation_Output"

## Step 3 — Confirm GPU and load CPSAM

Verify that a GPU is available, then load the CPSAM model. The first time this cell runs, Cellpose downloads ~1.15 GB of model weights from HuggingFace to `~/.cellpose/models/`. Subsequent runs use the cached weights.

In [ ]:
if not core.use_gpu():
    print("WARNING: No GPU detected. Runtime will be very long (8+ hours for 12-well).")
    print("Consider running on a GPU-equipped machine for production use.")
else:
    print("GPU confirmed. Proceeding with CPSAM segmentation.")

# Load model. Downloads on first run (~1.15 GB).
model = models.CellposeModel(gpu=True)


## Step 4 — Define the normalization helper

`normalize_image` applies Cellpose's per-region normalization across 1824-pixel sub-tiles, matching the preprocessing used by the Element Biosciences segmentation workflow.

In [4]:
def normalize_image(image, region_size=1824):
    image_norm = np.zeros_like(image, np.single)

    for xi in range(int(image.shape[1] / region_size)):
        for yi in range(int(image.shape[0] / region_size)):
            cropped = image[
                yi * region_size:(yi + 1) * region_size,
                xi * region_size:(xi + 1) * region_size,
            ]
            cropped = transforms.normalize_img(
                cropped.reshape(cropped.shape[0], cropped.shape[1], 1)
            ).reshape(cropped.shape[0], cropped.shape[1])
            image_norm[
                yi * region_size:(yi + 1) * region_size,
                xi * region_size:(xi + 1) * region_size,
            ] = cropped

    return image_norm


## Step 5 — Build the tile list from `RunParameters.json`

Read `RunParameters.json` to enumerate every well and tile in the run and build the `tile2well` map used by the segmentation loop. The cell prints the total tile count so you can confirm the workload before committing to Step 6.

In [ ]:
with open(os.path.join(run_directory, "RunParameters.json")) as f:
    run_parameters = json.load(f)

tile2well, tiles = {}, []
for well in run_parameters["Wells"]:
    for tile in well["Tiles"]:
        tile2well[tile["Name"]] = well["WellLocation"]
        tiles.append(tile["Name"])

print(f"Total tiles to process: {len(tiles)}")


## Step 6 — Segment every tile and write masks

Run CPSAM on each tile and write the cell and nuclear masks to `output_location/Well{well}/`. Unlike the Element Biosciences workflow, CPSAM uses a single model for every well, so no per-well model lookup is needed.

To monitor progress, watch for the rolling `Done: ...` lines. Each line corresponds to one tile fully processed and saved. See the **Runtime expectations** table at the bottom of the notebook for typical wall-clock times.

In [ ]:
flow_threshold      = 0.4
cellprob_threshold  = 0.0
tile_norm_blocksize = 0

print(f"Beginning segmentation across {len(tiles)} tiles")

for tile in tiles:
    well = tile2well[tile]
    os.makedirs(os.path.join(output_location, f"Well{well}"), exist_ok=True)

    cell_image    = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Cell-Membrane.tif")
    )
    nuclear_image = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Nucleus.tif")
    )
    actin_image   = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Actin.tif")
    )

    cell_image    = normalize_image(cell_image)
    nuclear_image = normalize_image(nuclear_image)
    actin_image   = normalize_image(actin_image)

    composite = np.zeros((cell_image.shape[0], cell_image.shape[1], 3))
    composite[:, :, 0] = cell_image
    composite[:, :, 1] = nuclear_image
    composite[:, :, 2] = actin_image

    print(f"Segmenting cell membrane \u2014 tile: {tile}")
    cell_mask, _, _ = model.eval(
        composite,
        batch_size=2,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
        resample=False,
    )
    cell_mask = cell_mask.astype(np.uint32)

    print(f"Segmenting nuclei \u2014 tile: {tile}")
    nuclear_mask, _, _ = model.eval(
        nuclear_image,
        batch_size=2,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
        resample=False,
    )
    binary_nuclei = nuclear_mask.copy()
    binary_nuclei[nuclear_mask > 0] = 1

    skimage.io.imsave(
        os.path.join(output_location, f"Well{well}", f"{tile}_Cell.tif"),
        cell_mask.astype(np.uint16),
    )
    skimage.io.imsave(
        os.path.join(output_location, f"Well{well}", f"{tile}_Nuclear.tif"),
        binary_nuclei.astype(np.uint8),
    )
    print(f"Done: {tile}")


## Reference

### Runtime expectations

CPSAM processes every tile in the run. CPU runtimes are prohibitive; the table below assumes a GPU.

| Plate format | Approximate tiles | GPU estimate |
| ------------ | ----------------- | ------------ |
| 1-well       | ~18 tiles         | ~30 minutes  |
| 12-well      | ~216 tiles        | ~6–10 hours |
| 48-well      | ~864 tiles        | ~24–36 hours |

### Output files

For each tile, the loop writes two files to your `output_location`:

- `{tile}_Cell.tif`: a `uint16` label mask where each unique integer represents one segmented cell.
- `{tile}_Nuclear.tif`: a `uint8` binary mask where `0` indicates no nucleus and `1` indicates a nucleus is present.

### Validation

CPSAM does not produce a built-in quality metrics table. Verify outputs visually or by comparing cell and nucleus counts against the baseline you established in [Interpret results and choose a model](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#interpret-results).

### Third-party tool disclaimer

CellposeSAM is provided by the [MouseLand open-source project](https://github.com/mouseland/cellpose) and is not affiliated with or endorsed by Element Biosciences. CPSAM has not been formally validated against AVITI24 cytoprofiling runs and results may vary. For CPSAM-specific issues, installation support, or model updates, refer to the [official MouseLand repository](https://github.com/mouseland/cellpose).

After all tiles finish, re-run Cells2Stats with `--segmentation` pointing at `output_location` to regenerate the cell table. See [Re-run Cells2Stats for cell assignment](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#cell-assignment).